# Media Manager Data Model  

The Database Model is based on the data feeds from:
* Trakt API
* The Movie DB API
    * Shows
    * Seasons
    * Episodes
    * Cast
    * Crew
    * Networks
    * Artwork URL's
* local configuration  
  
![alt text](../images/Media_Manager_DB.drawio.png)
  
    
The database is a MySQL Database called : **media_analytics_db** and it is hosted on a local Wamp Server Environment

I have 
* 6 Tables storing TMDB data
* 1 Table storing Trakt data
* 3 "Bridge" Tables to manage many-to-many relations (e.g. An Episode has many Actors (Person), and an Actor appears in many Episodes)  
* 1 Standalone Table storing Trakt Auth details (due to complexity of Trakt API Authorization)   
  
All Data tables are created with SQL Script [CREATE_DB.sql](../data/sql/CREATE_DB.sql)  
  
The Database uses Stored Procedures extensively for extracting, inserting, updating, and deleting records.   
There is a total of 19 Stored Procedures in use within this Database.
  
  All Stored Procedures are created with SQL Script [CREATE_SP.sql](../data/sql/CREATE_SP.sql)

***
## Database Generation Scripts  
A full set of scripts used to generate all tables/stored procedures/views is available under /data/sql/ folder ...
* Database and Tables : 
[CREATE_DB.sql](../data/sql/CREATE_DB.sql)
All Table CREATE scripts with Primary Keys, Foreign Keys, and Constraints
* Stored Procedures : 
[CREATE_SP.sql](../data/sql/CREATE_SP.sql)
All Stored Procedures
* Views : 
[CREATE_VIEW.sql](../data/sql/CREATE_VIEW.sql)

These scripts can be used to completely re-create the database and all components

***
## Data feeds into Database   
Unlike the "Book" database generated by Andrew Beatty demonstrated during his lectures, I was reluctant to allow users to be manually adding data records. 
In looking at the above DB Model, it should be clear that there is too much data to input.  
Secondly, with TMDB being a very robust, reliable, and up-to-date data source that has a freely available API ... **Data should be sourced from this Service not manual user inputs.**  
  
In regards to the number of TV Show records pulled from the TMDB API, it is totally impractical to be extracting all TV Shows.   
Estimates on the number of Show Datasets on TMDB range from 300,000 to 500,000 😮   
So, the number of tv show datasets extracted from TMDB will be dictated by the number of Shows I have watched previously (.i.e my personal Trakt History). As stated in other notebook ([trakt_api](trakt_api.ipynb)), I have been using Trakt for several years to manage my watch history, and I have an extensive history of using this service.   
**I will use their API to extract my personal media watch history, and use this as a baseline list of TV Shows to add to database.**

***
## Dataset Size and Growth

As of mid April 2026, my Trakt account has a record of me watching 167 Shows to date over the past few decades.  
* **For each TV Show that I have watched, started to watch, or not fully watched ... I extract all Season/Episode Data**.  
  
This translates into the following TMDB records that were extracted from the TMDB Service

* 167 TV Shows
* 776	Seasons
* 1580 Episodes	
* 2150 Distinct People (Cast/Crew)
* 11561 Episode Cast (Characters)	
* 2631 Episode Crew (Director/Writer/Producer)
* 46 Networks	
* 10253 Image URL's

Loading this all this data into my Database did cause a few issues with regards to efficient design and performance.  
It takes up to 5 hours to make all the necessary API Calls and load responses into memory before loading into database.  
In order to avoid hitting limits in the Free Tier of the TMDB API, I had added a 1.0 second delay between API calls.  

* It is my intention to fully load this TMDB data once a week to keep my Data up-to-date.  
* Each night the Trakt Table will be refreshed from the Trakt service.  
    * After this nightly refresh, python services will monitor for any new Shows that are found in this Trakt Table not yet in TMDB Tables, then this show's TMDB data is loaded.  
      


***
<img src="../images/trakt_mobile.png" alt="Panda" width="300">        
</br></br></br>
<img src="../images/trakt_addon.png" alt="Panda" width="350">  
  
As I watch TV Shows (and record on Trakt with Mobile App or Service Add-ons), my Database will be updated accordingly each night.